# 1. Recolección de Datos - API de SpaceX

**Objetivo:** obtener el historial de lanzamientos del Falcon 9 usando la API pública de SpaceX (`api.spacexdata.com`), para más adelante predecir si la primera etapa (booster) aterriza con éxito.

**Flujo de trabajo:**

1. Pedir a la API la lista de lanzamientos pasados (`GET /v4/launches/past`).
2. Quedarnos con las columnas relevantes: `rocket`, `payloads`, `launchpad`, `cores`, `flight_number`, `date_utc`.
3. Filtrar solo lanzamientos con **un** core y **un** payload (simplifica el análisis: son la gran mayoría de los Falcon 9).
4. Filtrar por fecha (hasta 2020-11-13, corte usado por la consigna del curso).
5. Por cada `rocket_id`, `launchpad_id`, `payload_id` y `core_id`, pedir el detalle a la API (`/rockets/{id}`, `/launchpads/{id}`, `/payloads/{id}`, `/cores/{id}`) y armar las columnas finales (nombre del cohete, sitio de lanzamiento, masa de la carga, órbita, resultado del aterrizaje, etc.).
6. Quedarnos solo con lanzamientos de **Falcon 9** (se descarta el Falcon 1, que no tiene aterrizaje).
7. Completar valores faltantes de `PayloadMass` con la media de la columna.
8. Exportar el resultado a `data/raw/dataset_part_1.csv`.

> **Nota:** este notebook necesita conexión a internet para llamar a la API pública de SpaceX.

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)

## Funciones auxiliares

Cada función recibe el DataFrame filtrado y llama a un endpoint distinto de la API para completar los datos, guardando los resultados en listas globales.

In [ ]:
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []


def getBoosterVersion(data):
    """Llama a /v4/rockets/{id} para obtener el nombre del cohete (ej. 'Falcon 9')."""
    for x in data['rocket']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()
            BoosterVersion.append(response['name'])


def getLaunchSite(data):
    """Llama a /v4/launchpads/{id} para obtener el sitio de lanzamiento y sus coordenadas."""
    for x in data['launchpad']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])


def getPayloadData(data):
    """Llama a /v4/payloads/{id} para obtener la masa de la carga y la órbita objetivo."""
    for load in data['payloads']:
        if load:
            response = requests.get("https://api.spacexdata.com/v4/payloads/" + str(load)).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])


def getCoreData(data):
    """Llama a /v4/cores/{id} para obtener datos del booster y arma la columna Outcome
    (resultado del aterrizaje: exito/fracaso + tipo: ASDS/RTLS/Ocean)."""
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get("https://api.spacexdata.com/v4/cores/" + str(core['core'])).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

## Paso 1: pedir los lanzamientos pasados a la API

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print("Status code:", response.status_code)

data = pd.json_normalize(response.json())
data.head()

## Paso 2: nos quedamos con las columnas relevantes

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data.head()

## Paso 3: filtramos lanzamientos de un solo core y un solo payload

In [ ]:
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Extraemos el único elemento de cada lista
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

data.shape

## Paso 4: filtramos por fecha (hasta 2020-11-13)

In [ ]:
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

data.shape

## Paso 5: enriquecemos los datos llamando a la API por cada id

Esto puede tardar un poco: hace una llamada HTTP por cada cohete/sitio/carga/booster.

In [ ]:
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)

launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude,
}

df = pd.DataFrame(launch_dict)
df.head()

## Paso 6: nos quedamos solo con Falcon 9

El Falcon 1 no tiene intento de aterrizaje de la primera etapa, así que lo descartamos y renumeramos los vuelos.

In [ ]:
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].copy()
data_falcon9['FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9.head()

## Paso 7: valores faltantes

Revisamos qué columnas tienen nulos y completamos `PayloadMass` con la media de la columna.

In [ ]:
data_falcon9.isnull().sum()

In [ ]:
mean_payload_mass = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(mean_payload_mass)

data_falcon9['PayloadMass'].isnull().sum()

## Paso 8: exportamos el dataset

In [ ]:
data_falcon9.to_csv('../data/raw/dataset_part_1.csv', index=False)
data_falcon9.shape

## Resumen

Terminamos con un dataset de lanzamientos de Falcon 9 con una fila por lanzamiento (single-core, single-payload),
con columnas de cohete, carga útil, sitio de lanzamiento y el resultado del aterrizaje (`Outcome`), listo para
combinarlo con los datos de web scraping y pasar a la etapa de **Data Wrangling** (notebook 03).